In [ ]:
# /Users/rafaelfelix/.cache/huggingface/lerobot//rafafelixphd/dataset-trial-1/data/chunk-000/file-001.parquet

import pandas as pd
from glob import glob

# parquet_path = "/Users/rafaelfelix/.cache/huggingface/lerobot//rafafelixphd/dataset-trial-1/meta/tasks.parquet"
# parquet_path = "/Users/rafaelfelix/.cache/huggingface/lerobot//rafafelixphd/dataset-trial-1/data/chunk-000/file-001.parquet"
parquet_path = "/Users/rafaelfelix/.cache/huggingface/lerobot//rafafelixphd/dataset-trial-1/meta/episodes/chunk-000/file-000.parquet"
df = pd.read_parquet(parquet_path)
sample = df.iloc[0]

In [ ]:
parquet_path = "/Users/rafaelfelix/.cache/huggingface/lerobot//rafafelixphd/dataset-trial-1/meta/episodes/chunk-000/file-000.parquet"

episode_index	tasks	length	data/chunk_index	data/file_index	dataset_from_index	dataset_to_index	videos/observation.images.front/chunk_index	videos/observation.images.front/file_index	videos/observation.images.front/from_timestamp	...	stats/task_index/mean	stats/task_index/std	stats/task_index/count	stats/task_index/q01	stats/task_index/q10	stats/task_index/q50	stats/task_index/q90	stats/task_index/q99	meta/episodes/chunk_index	meta/episodes/file_index
0	0	[Grab the red cylinder]	401	0	0	0	401	0	0	0.000000	...	[0.0]	[0.0]	[401]	[3.9999999999994176e-16]	[3.999999999999417e-15]	[1.9999999999997088e-14]	[3.599999999999476e-14]	[3.9599999999994235e-14]	0	0
1	1	[Grab the red cylinder]	398	0	0	401	799	0	0	13.366667	...	[0.0]	[0.0]	[398]	[3.9999999999994176e-16]	[3.999999999999417e-15]	[1.9999999999997088e-14]	[3.599999999999476e-14]	[3.9599999999994235e-14]	0	0

In [ ]:
df.columns

In [ ]:
sample

# Changing file
> parquet_path = "/Users/rafaelfelix/.cache/huggingface/lerobot//rafafelixphd/dataset-trial-1/meta/episodes/chunk-000/file-000.parquet"
> 


In [ ]:
pfilesdd

In [5]:
import pandas as pd
from glob import glob

pfiles = glob("/Users/rafaelfelix/.cache/huggingface/lerobot/rafafelixphd/dataset-trial-1/meta/episodes/chunk-000/*.parquet")
parquet_path = pfiles[0]
# print(parquet_path)
# df = pd.read_parquet(parquet_path)

print("\n".join(pfiles))


/Users/rafaelfelix/.cache/huggingface/lerobot/rafafelixphd/dataset-trial-1/meta/episodes/chunk-000/file-006.parquet
/Users/rafaelfelix/.cache/huggingface/lerobot/rafafelixphd/dataset-trial-1/meta/episodes/chunk-000/file-007.parquet
/Users/rafaelfelix/.cache/huggingface/lerobot/rafafelixphd/dataset-trial-1/meta/episodes/chunk-000/file-005.parquet
/Users/rafaelfelix/.cache/huggingface/lerobot/rafafelixphd/dataset-trial-1/meta/episodes/chunk-000/file-004.parquet
/Users/rafaelfelix/.cache/huggingface/lerobot/rafafelixphd/dataset-trial-1/meta/episodes/chunk-000/file-001.parquet
/Users/rafaelfelix/.cache/huggingface/lerobot/rafafelixphd/dataset-trial-1/meta/episodes/chunk-000/file-011.parquet
/Users/rafaelfelix/.cache/huggingface/lerobot/rafafelixphd/dataset-trial-1/meta/episodes/chunk-000/file-008.parquet
/Users/rafaelfelix/.cache/huggingface/lerobot/rafafelixphd/dataset-trial-1/meta/episodes/chunk-000/file-009.parquet
/Users/rafaelfelix/.cache/huggingface/lerobot/rafafelixphd/dataset-trial

print("\n".join(list(df.columns)))

In [4]:
import pyarrow.parquet as pq

def re_write_parquet(filepath):
    table = pq.read_table(filepath)

    # 1. Update Video Pointers
    for video_key in ['chunk_index', 'file_index', 'from_timestamp', 'to_timestamp']:
        src_label = f'videos/observation.images.front/{video_key}'
        if src_label in table.column_names:
            src_col = table.column(src_label)
            # Matching your folder naming: observation.images.left / .right
            table = table.append_column(f'videos/observation.images.left/{video_key}', src_col)
            table = table.append_column(f'videos/observation.images.right/{video_key}', src_col)
            table = table.remove_column(table.schema.get_field_index(src_label))

    # 2. Update Stats
    for stats_key in ['min', 'max', 'mean', 'std', 'count', 'q01', 'q10', 'q50', 'q90', 'q99']:
        src_label = f'stats/observation.images.front/{stats_key}'
        if src_label in table.column_names:
            src_col = table.column(src_label)
            table = table.append_column(f'stats/observation.images.left/{stats_key}', src_col)
            table = table.append_column(f'stats/observation.images.right/{stats_key}', src_col)
            table = table.remove_column(table.schema.get_field_index(src_label))

    pq.write_table(table, filepath)

In [6]:
for filename in pfiles:
    re_write_parquet(filename)

In [ ]:
for video_key in ['chunk_index', 'file_index', 'from_timestamp', 'to_timestamp']:
    df[f'videos/observation.cameras.left/{video_key}'] = df[f'videos/observation.images.right/{video_key}'] = df[f'videos/observation.images.front/{video_key}']
    del df[f'videos/observation.images.front/{video_key}']

for stats_key in ['min', 'max', 'mean', 'std', 'count', 'q01', 'q10', 'q50', 'q90', 'q99']:
    df[f'stats/observation.images.left/{stats_key}'] = df[f'stats/observation.images.right/{stats_key}'] = df[f'stats/observation.images.front/{stats_key}']
    del df[f'stats/observation.images.front/{stats_key}']

In [ ]:
df.to_parquet(parquet_path, index=False)